## 核心作用
with as 语句用于高效管理资源（如文件操作、数据库连接、锁机制等），它能自动分配并在离开代码块时**自动释放资源**，即使发生异常也能确保安全。例如，使用 with as 操作已经打开的文件对象（本身就是上下文管理器），无论期间是否抛出异常，都能保证 with as 语句执行完毕后自动关闭已经打开的文件。

```
with 表达式 [as target]： #此格式中，用 [] 括起来的部分可以使用，也可以省略。
    代码块
```
- 执行流程：
    1. 执行 表达式，并调用其内部的 __enter__() 方法。
    2. __enter__() 的返回值会赋值给 as 后面的 变量。
    3. 执行缩进的 代码块。无论代码块正常结束还是抛出异常，都会自动调用对象的 __exit__() 方法完成清理工作（如关闭文件）。
- target 参数用于指定一个变量，该语句会将 expression 指定的结果保存到该变量中。
- with as 语句中的代码块如果不想执行任何语句，可以直接使用 pass 语句代替。

In [1]:
class Sample(object):
    def __enter__(self):
        print("In __enter__()")
        return "Foo"

    def __exit__(self, type, value, trace):
        print("In __exit__()")


def get_sample():
    return Sample()


with get_sample() as sample:
    print("sample:", sample)


print(Sample)    # 这个表示类本身   <class '__main__.Sample'>
print(Sample())  # 这表示创建了一个匿名实例对象 <__main__.Sample object at 0x00000259369CF550>

In __enter__()
sample: Foo
In __exit__()
<class '__main__.Sample'>


## 步骤分析:
1. 调用get_sample()函数，返回Sample类的实例;
2. 执行Sample类中的__enter__()方法，打印"In__enter_()"字符串，并将字符串“Foo”赋值给as后面的sample变量;
3. 执行with-block码块，即打印"sample: %s"字符串，结果为"sample: Foo"
4. 执行with-block码块结束，返回Sample类，执行类方法__exit__()。因为在执行with-block码块时并没有错误返回，所以type,value,trace这三个arguments都没有值。直接打印"In__exit__()"

## 常见应用场景
### 文件操作
不用 with 时，如果读取文件出错，可能导致文件句柄无法关闭

In [4]:
# 传统写法
f = open('../practise/1-两变量交换值.py', 'r', encoding='utf-8')
try:
    data = f.read()
    print(data)
finally:
    f.close()
print("="*20)

# 使用with as 简化后:
with open('../practise/1-两变量交换值.py', 'r', encoding='utf-8') as f:
    data = f.read()
    print(data)

a=1
b=2
b,a=a,b
print(a,b)
a=1
b=2
b,a=a,b
print(a,b)


### 锁机制（多线程）
在多线程编程中，用它来自动获取和释放锁：

In [ ]:
import threading

lock = threading.Lock()

with lock:
    # 自动执行 lock.acquire()
    print("安全访问共享资源")
# 离开代码块自动执行 lock.release()

### 自定义上下文管理
只要一个类实现了 __enter__() 和 __exit__() 两个魔术方法，就可以用 with...as 操作它。

In [ ]:
class MyResource:
    def __enter__(self):
        print("资源已创建/打开")
        return self  # 返回的对象会赋值给 as 后的变量

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("资源已自动释放/关闭")
        # 返回 False（或默认不返回）会把异常正常向上抛出
        # 返回 True 可以吞掉异常

# 使用自定义类
with MyResource() as res:
    print("正在使用资源")


## 如何在 with 中同时管理多个资源
如果需要在同一个 with 块中同时管理多个资源（例如：同时打开源文件和目标文件以进行内容复制），有两种最常用的优雅写法。

### 方法一：在一行中用逗号隔开（最常用、最推荐）
Python 允许你在同一个 with 语句中连续编写多个表达式，中间用逗号 , 分隔。
1. 基础语法
```python
with 资源A as a, 资源B as b:
    # 同时使用 a 和 b 进行操作
```
2. 实战示例（文件复制）
从 Python 3.10 开始，支持在 with 后面加圆括号 () 进行多行美化，这样当资源路径很长时，代码依然非常清晰：

In [ ]:
# 推荐写法（Python 3.10+ 支持圆括号换行）
with (
    open("source.txt", "r", encoding="utf-8") as src,
    open("dest.txt", "w", encoding="utf-8") as dst
):
    content = src.read()
    dst.write(content)

# 如果是 Python 3.9 及以下版本，则不能加圆括号，需在一行写完或使用反斜杠 \ 换行：
with open("source.txt", "r") as src, open("dest.txt", "w") as dst:
    dst.write(src.read())

执行与释放顺序进入时（__enter__）：
- 按照从左到右的顺序依次初始化（先 src 再 dst）。
- 退出时（__exit__）：按照从右到左的相反顺序依次释放（先关闭 dst 再关闭 src）。这种“后进先出”的机制确保了资源依赖的安全性。

### 方法二：在一行中用逗号隔开（最常用、最推荐）
如果你需要管理的资源数量是动态变化的（例如：根据用户输入，同时打开不确定数量的 5 个或 10 个文件），在一行中写死显然不适用。这时可以使用 contextlib.ExitStack。ExitStack 就像一个可以随时放入资源的“栈”，无论中途发生什么异常，进入该栈的所有资源最后都会被自动且正确地关闭。

In [ ]:
from contextlib import ExitStack

file_paths = ["file1.txt", "file2.txt", "file3.txt"]

with ExitStack() as stack:
    # 动态将多个文件注册到上下文中
    files = [stack.enter_context(open(path, "w", encoding="utf-8")) for path in file_paths]

    # 此时 files 列表中包含 3 个已打开的文件对象
    files[0].write("写入第一个文件\n")
    files[1].write("写入第二个文件\n")
    files[2].write("写入第三个文件\n")

# 离开 with 块后，所有 3 个文件都会被自动安全关闭
